In [1]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow # Colab 환경에서 이미지 표시를 위한 함수 임포트
import time # 출력 속도를 조절하기 위한 time 모듈 임포트


# --- 1. 초기 설정 및 변수 정의 ---
# NOTE: Colab에서 이 코드를 실행하기 전에, 'newyork.mp4' 파일을 Colab 환경에 업로드해야 합니다.
video_path = '/content/drive/MyDrive/rokey/AI/application/4/newyork.mp4'
cap = cv2.VideoCapture(video_path) # 비디오 캡처 객체 생성


# 비디오 파일이 제대로 열렸는지 확인
if not cap.isOpened():
    print("오류: 비디오 파일을 열 수 없습니다. 파일을 업로드했는지 확인하세요.")
    exit()


Optical Flow?
- 연속된 두 프레임 사이에서 픽셀이 어디로 움직였는지 계산
- 물체 속도 방향
- 자율주행, 드론, 동작인식

In [4]:
# 프레임 재생 속도 조절을 위한 딜레이 (1000ms = 1s / 30fps = 1000ms ÷ 30 ≈ 33.3ms)
# Colab에서는 실시간 딜레이 대신 time.sleep()으로 대체되지만, 변수는 주석을 위해 유지합니다.

delay = int(1000/30) # 1000ms ÷ 30frame = 33.3ms
delay
# real-time 실시간 영상에서 부드러운 재생을 원하기 때문에

# 추적 경로를 그리기 위한 랜덤 색상(200개의 코너점에 대응하는 색상)
# np.random.randint(0, 255, (200,3)) # randn은 평균0 분산1이라서 int 아님
# 0-255 범위에서 200개 코너점의 3개 채널

color = np.random.randint(0, 255, (200, 3))

lines = None # 추적선(이동경로) 그릴 이미지 저장 변수(초기화: 첫 프레임에서 진행)
prevlmg = None # previous image : 이전 프레임(grayscale image)

# calcOpticalFlowPyrLK() 중지 요건 설정 (Termination Criteria) = 추적 종료 조건
# LK 알고리즘이 한 프레임 안에서 “특정 점이 얼마나 이동했는지 계산하는 반복 연산을 언제 멈출지” 정하는 조건
# (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 최대 반복 횟수(10), 오차 임계값(0.03))
# cv2.TERM_CRITERIA_EPS : 움직임이 0.03 px 미만 >> 수렴 (error 오차가 충분히 작아지면 멈춰)
# cv2.TERM_CRITERIA_COUNT : 최대 반복 횟수
# 반복 횟수가 10번을 넘으면 멈춰
# 이동 변화량이 0.03보다 작아도 멈춰
# 둘 중에 하나라도 만족하면 종료(EPS 또는 COUNT)
termcriteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03) # 추적 종료 조건
termcriteria # (3, 10, 0.03)
# 3 → criteria_type (종료 기준 종류:3=두 조건 모두 사용) / 10 → max_count (최대 반복 횟수) / 0.03 → epsilon (얼마나 변화가 작아지면 멈출지)

frame_count = 0
MAX_FRAMES_TO_PROCESS = 150 # 최대 150 프레임만 처리
DISPLAY_EVERY_N_FRAMES = 20 # 20 프레임마다 결과 출력

print(f"광학 흐름 추적 시작 (최대 {MAX_FRAMES_TO_PROCESS} 프레임, {DISPLAY_EVERY_N_FRAMES} 프레임마다 출력)...")


광학 흐름 추적 시작 (최대 150 프레임, 20 프레임마다 출력)...


In [ ]:
# 예제
'''
p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, gray, p0, None,
                                      winSize=(15,15),
                                      maxLevel=2,
                                      criteria=termcriteria)

cv2.calc0pticalFlowPyrLK(루카스-카나데 광학 흐름) 물체 이동 위치 추적
- old_gray : 이전 프레임(흑백)
- gray : 현재 프레임(흑백)
- p0 : 이전 프레임에서 추적할 점들
- None : 초기 예측 값 없음(기본 설정) 원래는 p1(예상 이동 위치)을 초기값으로 줄 수 있는데 보통 None으로 둠 → OpenCV가 알아서 계산하라는 뜻.
- winSize : 점 주변에서 추적할 영역 크기 (15 x 15 주변 픽셀을 묶어서 비교, 작으면 → 계산 빨라지지만 불안정함, 크면 → 안정적이지만 느림)
- maxLevel : 피라미드 단계 수 (=멀리 움직인 점도 잡아줌)
Level 0: 원본
Level 1: 절반 크기
Level 2: 그보다 더 작은 이미지
maxLevel=2 → 3개 이미지(0,1,2 level)에서 추적→ 멀리 움직인 점도 잘 찾기 위함
각 레벨마다 15×15 크기의 지역창(window)을 이용해 LK를 따로 수행한다 : 레벨 2 → 1 → 0 순으로 점점 더 정밀하게 보정해가는 구조
- criteria : 계산 반복을 언제 멈출지 조건

- p1 : 추적된 새로운 좌표값 position
- st : 추적 성공 여부(1: 성공, 0: 실패) status
- err : 추적 오류(오차 정도) error

p0 (100, 200) 서있는 상태
p1 (50, 100) 오리걸음 하는 상태

st = 1 # 성공 (잘 추적했네)
err = 0.01 # 오차 거의 없네 (기준인 0.03 미만이니깐)


'''

In [7]:
# 비디오 처리

while cap.isOpened() and frame_count < MAX_FRAMES_TO_PROCESS:
  ret, frame = cap.read()

  if not ret:
    break

  # 현재 프레임(사진) 복사 >> 추적결과 그릴 이미지 준비
  img_draw = frame.copy()
  # optical flow 계산을 위해 현재 프레임을 grayscale로 변환
  gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

  # 최초 프레임 처리(추적 시작)
  if prevlmg is None:
    prevlmg = gray  # 현재 gray 이미지를 이전 이미지로 저장
    # 추적선을 그릴 검은색 배경 이미지 생성(원본 프레임과 동일 크기)
    lines = np.zeros_like(frame) #검은색이 0니까 zeros_like으로 동일 크기 만듦
    # Shi-Tomasi 알고리즘으로 추적
    # (이전 이미지, 최대 코너점 수, 품질 임계값(0.01), 최소거리)
    ''' 여기 추적할 점들(goodFeaturesToTrack) 찾아놔! '''
    prevPt = cv2.goodFeaturesToTrack(prevlmg, 100, 0.01, 10) # prevPt: 코너 점 목록
  # 두번째 프레임 이후 처리(추적 진행 중)
  else:
    ''' 직전 프레임의 점들이 이번 프레임에서는 어디로 움직였는지 추적(calcOpticalFlowPyrLK)해! '''
    nextlmg = gray # 현재 gray 이미지를 '다음 이미지로' 저장
    nextPt, status, err = cv2.calcOpticalFlowPyrLK(prevlmg, nextlmg,
                             prevPt, None, criteria=termcriteria)

    # 추적에 성공한 코너점(status==1) 선별
    # status == 1인 점만 남긴다는 의미
    # 예) status = [1, 1, 0, 1, 0, 1]
    prevMv = prevPt[status==1] # 이전 프레임에서 추적 성공한 점
    nextMv = nextPt[status==1] # 현재 프레임에 대응하는 점

    # 추적 성공한 모든 쌍에 대해 반복
    for i, (p, n) in enumerate(zip(prevPt, nextPt)):
      # 코너점 좌표 추출(배열 구조 해제)
      px, py = p.ravel()
      nx, ny = n.ravel()

      # 이전 코너(p)와 새로운 코너(n) 사이에 추적선 그리기(lines 이미지에 누적)
      cv2.line(lines, (int(px), int(py)), (int(nx), int(ny)), color[i % len(color)].tolist(), 2)
      # 시작점: 이전 위치(px, py), 끝점(nx, ny)
      # color[i % len(color)].tolist() : 코너점 i에서 할당된 랜덤 색상

      # 새로운 코너(n)에 원형 점 그리기(img_draw 이미지에 매 프레임마다 표시)
      cv2.circle(img_draw, (int(nx), int(ny)), 2, color[i % len(color)].tolist(), -1)

    # 누적된 추적선이 그려진 lines 이미지와 현재 프레임(img_draw)을 합성
    # >> 추적 경로가 비디오 프레임 위에 나타남
    img_draw = cv2.add(img_draw, lines)

    # 다음 루프 위해 현재 프레임(사진)과 코너점 >> 이전 변수로 이동
    prevlmg = nextlmg
    prevPt = nextPt

    prevPt = nextMv.reshape(-1, 1, 2)
    # prevPt를 nextMv 형태로 맞춰줘
    # (N, 2) >> (N, 1, 2)
    # -1 : 자동계산

    # 원래 좌표 형식 [[x,y],[x,y],[x,y]] shape(3,2) >> (N, 2)
    # OpenCV optical Flow 함수 입력 [[[x,y],[x,y],[x,y]]] shape (3,1,2) >>(N,1,2)
    # 각 좌표를 2차원으로 입력해줘

    # Colab 출력: 특정 간격의 프레임만 표시
    if frame_count % DISPLAY_EVERY_N_FRAMES == 0:
        print(f"\n--- Frame {frame_count} 광학 흐름 결과 ---")
        cv2_imshow(img_draw) # 추적 결과 프레임 표시
        time.sleep(1) # 출력이 빠르게 지나가는 것을 방지하기 위해 잠시 대기




    # 키 입력 처리 (원본 코드에 있던 부분. Colab에서는 작동하지 않으므로,
    # 프레임 수 제한으로 대체하며 주석 처리만 해둡니다.)
    # key = cv2.waitKey(delay)
    # if key == 27:    # ESC 키 (ASCII 27): 루프 종료
    #     break

    # elif key == 8:  # Backspace 키 (ASCII 8): 추적 이력 지우기
    #     prevImg = None


    frame_count += 1


# --- 3. 종료 및 정리 (수정 완료) ---
# 모든 OpenCV 창 닫기 (Colab에서는 필요 없음)
# cv2.destroyAllWindows()
# 비디오 캡처 객체 반환 및 해제
cap.release()


Output hidden; open in https://colab.research.google.com to view.